# imprecision-bench: Benchmark Orientation Guide

This notebook introduces the **imprecision-bench** dataset and shows what current language and vision-language models do on it. The benchmark tests a simple pragmatic question: does a model adjust *how precisely* it reports a time depending on social context — saying "8:30" to a police officer but "around half past eight" to a neighbor? Humans do this naturally. Current models largely don't.

**Two tasks:**
- **Task 1 (production):** given a clock image or text description + scenario, complete "It happened ___."  
- **Task 2 (motive elicitation):** given the production + context, explain why that wording was chosen.

**Dataset:** 475 human productions × 12 clock states × 2 contexts (police / neighbor)  
**Source:** Mühlenbernd & Solt (2022), *Linguistics Vanguard*, DOI: 10.1515/lingvan-2022-0035

## 1. Setup

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from scipy.stats import wasserstein_distance
from IPython.display import display, Image as IPImage
import io

HF_REPO_ID = "RolandM/imprecision-bench"
RESULTS_CSV = "results.csv"

# Models included in this notebook
MODELS = {
    "gpt_4o_mini":             "GPT-4o mini",
    "claude_haiku_4_5":        "Claude Haiku 4.5",
    "google_gemini_2_5_flash": "Gemini 2.5 Flash",
}
MODEL_COLORS = {
    "gpt_4o_mini":             "#74aa9c",
    "claude_haiku_4_5":        "#d97757",
    "google_gemini_2_5_flash": "#4285f4",
}

### API keys

To re-run or extend the evaluation, copy `.env.example` to `.env` and fill in your keys:

```
OPENAI_API_KEY=...       # for gpt-4o, gpt-4o-mini
ANTHROPIC_API_KEY=...    # for claude-* models
OPENROUTER_API_KEY=...   # for google/gemini-* via OpenRouter
TOGETHER_API_KEY=...     # for open-weight models via Together.ai
```

Then run:
```bash
python evaluate.py --model gpt-4o-mini --rows 50   # pilot: first 50 rows
python evaluate.py --model gpt-4o-mini --full       # full: all 475 rows
```

## 2. The Benchmark Data

In [ ]:
ds = load_dataset(HF_REPO_ID, split="train")
print(f"Dataset: {len(ds)} rows")

context_names  = ds.features["context"].names
stimulus_names = ds.features["stimulus_type"].names

df_full = ds.to_pandas()
df_full["context_str"]  = df_full["context"].map(lambda x: context_names[x])
df_full["stimulus_str"] = df_full["stimulus_type"].map(lambda x: stimulus_names[x])

print("\nCondition breakdown:")
print(df_full.groupby(["target_time", "context_str"]).size().unstack())

In [ ]:
# Show a sample row: clock image, description, and human response
sample = df_full[df_full["context_str"] == "police"].iloc[0]

print(f"Target time     : {sample['target_time']}")
print(f"Context         : {context_names[sample['context']]}")
print(f"Stimulus type   : {stimulus_names[sample['stimulus_type']]}")
print(f"Clock description: {sample['clock_description']}")
print(f"Prompt          : {sample['prompt']}")
print(f"Human production: {sample['production']}")
print()
print("Clock image:")
img_buf = io.BytesIO()
sample["clock_image"].save(img_buf, format="PNG")
display(IPImage(data=img_buf.getvalue(), width=200))

## 3. Results Overview

Pre-computed results are stored in `results.csv`. Each model adds three columns:
- `{model}_task1_image` — production from clock *image*
- `{model}_task1_text` — production from clock *text description*
- `{model}_task2` — motive explanation

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f"Results file: {len(results)} rows, {len(results.columns)} columns")

coverage = {}
for key, label in MODELS.items():
    col = f"{key}_task1_text"
    if col in results.columns:
        coverage[label] = results[col].notna().sum()

print("\nModel coverage (Task 1 text, non-null rows):")
for label, n in coverage.items():
    print(f"  {label}: {n} / {len(results)}")

## 4. Task 1 — Clock Reading: Image vs. Text

The benchmark includes two versions of Task 1: one where the model sees the **clock image**, and one where it sees a **textual description** of the same clock. This lets us separate visual grounding ability from linguistic pragmatics.

In [ ]:
# Show raw outputs side by side for the first 10 rows with data
sample_rows = results.dropna(subset=["gpt_4o_mini_task1_image"]).head(10)

display_cols = ["target_time", "context"]
for key in MODELS:
    img_col  = f"{key}_task1_image"
    text_col = f"{key}_task1_text"
    if img_col in results.columns:
        display_cols += [img_col, text_col]

pd.set_option("display.max_colwidth", 35)
display(sample_rows[display_cols].reset_index(drop=True))

In [ ]:
# Modality gap: heuristic — does the response contain the correct hour digit?
# A rough proxy for clock-reading accuracy on the image task.
def contains_correct_hour(response, target_time):
    """Check if the response mentions the correct hour (8 for all conditions here)."""
    if pd.isna(response):
        return None
    return "8" in str(response) or "eight" in str(response).lower()

pilot = results.dropna(subset=["gpt_4o_mini_task1_image"]).copy()

print("Correct hour mentioned (image task vs text task):")
print(f"{'Model':<35} {'Image':>8} {'Text':>8}")
print("-" * 55)
for key, label in MODELS.items():
    img_col  = f"{key}_task1_image"
    text_col = f"{key}_task1_text"
    if img_col not in pilot.columns:
        continue
    img_acc  = pilot[img_col].apply(lambda r: contains_correct_hour(r, None)).mean()
    text_acc = pilot[text_col].apply(lambda r: contains_correct_hour(r, None)).mean()
    print(f"{label:<35} {img_acc:>7.0%} {text_acc:>7.0%}")

## 5. Task 1 — Pragmatic Shift: Police vs. Neighbor

The core question: does a model produce *more precise* times for a police officer than for a neighbor? Humans do — they round in casual contexts and give exact times in formal ones.

In [ ]:
# Simple proxy for precision: does the response contain ":" (digital time like "8:30")?
# Imprecise responses tend to use words ("half past eight", "around eight").
def is_digital(response):
    if pd.isna(response):
        return None
    return ":" in str(response)

pilot = results.dropna(subset=["gpt_4o_mini_task1_text"]).copy()

print("Rate of digital time format (e.g. \"8:30\") — text task:")
print(f"{'Model':<35} {'Police':>8} {'Neighbor':>8} {'Shift':>8}")
print("-" * 65)
for key, label in MODELS.items():
    col = f"{key}_task1_text"
    if col not in pilot.columns:
        continue
    pilot["digital"] = pilot[col].apply(is_digital)
    police   = pilot[pilot["context"] == "police"]["digital"].mean()
    neighbor = pilot[pilot["context"] == "neighbor"]["digital"].mean()
    shift    = police - neighbor
    print(f"{label:<35} {police:>7.0%} {neighbor:>7.0%} {shift:>+7.0%}")

# Human baseline
df_full["digital"] = df_full["production"].apply(lambda r: is_digital(r))
h_police   = df_full[df_full["context_str"] == "police"]["digital"].mean()
h_neighbor = df_full[df_full["context_str"] == "neighbor"]["digital"].mean()
print(f"{'Human baseline':<35} {h_police:>7.0%} {h_neighbor:>7.0%} {h_police - h_neighbor:>+7.0%}")

## 6. Task 2 — Motive Elicitation (Sample)

Task 2 asks the model to explain *why* a given time expression was chosen. This is not yet scored quantitatively — it is an open contribution opportunity.

In [ ]:
# Show three Task 2 examples for each model
sample = results.dropna(subset=["gpt_4o_mini_task2"]).head(3)
for _, row in sample.iterrows():
    print(f"[{row['context']} | {row['target_time']}]")
    print(f"  Human:   {row['human_production']}")
    for key, label in MODELS.items():
        col = f"{key}_task2"
        if col in row and pd.notna(row[col]):
            print(f"  {label[:25]}: {str(row[col])[:120]}")
    print()

## 7. Full Analysis (475 rows)

The following cells use the complete 475-row dataset to quantify the two headline findings: the **modality gap** and the **pragmatic shift failure**.

In [ ]:
# ── Helper: approximate time-correctness check ────────────────────────────────
# For each response, check whether it conveys the target time (exactly or via
# common word forms). Used to score the image task vs the text task.

MINUTE_WORDS = {
    25: ["twenty-five", "twenty five"],
    26: ["twenty-six", "twenty six"],
    27: ["twenty-seven", "twenty seven"],
    28: ["twenty-eight", "twenty eight"],
    29: ["twenty-nine", "twenty nine"],
    30: ["thirty", "half"],
    31: ["thirty-one", "thirty one"],
    32: ["thirty-two", "thirty two"],
    33: ["thirty-three", "thirty three"],
    34: ["thirty-four", "thirty four"],
    35: ["thirty-five", "thirty five"],
}

def time_correct(response, target_time):
    """True if the response conveys the target time (digit or word form)."""
    if pd.isna(response):
        return None
    r = str(response).lower()
    # Require correct hour
    if "8" not in r and "eight" not in r:
        return False
    # Range condition (e.g. 8:26-8:34): accept any number 26-34 or range language
    if "-" in str(target_time):
        return (
            any(str(m) in r for m in range(26, 35))
            or any(w in r for m in range(26, 35) for w in MINUTE_WORDS.get(m, []))
            or "between" in r
        )
    # Exact condition: check digit or word form of target minute
    minute = int(str(target_time).split(":")[1])
    return str(minute) in r or any(w in r for w in MINUTE_WORDS.get(minute, []))


# Compute accuracy for image and text tasks across all 475 rows
acc_rows = []
for key, label in MODELS.items():
    img_col  = f"{key}_task1_image"
    text_col = f"{key}_task1_text"
    df = results.dropna(subset=[img_col, text_col]).copy()
    img_acc  = df.apply(lambda r: time_correct(r[img_col],  r["target_time"]), axis=1).mean()
    text_acc = df.apply(lambda r: time_correct(r[text_col], r["target_time"]), axis=1).mean()
    acc_rows.append({"model": label, "key": key, "image": img_acc, "text": text_acc, "n": len(df)})

acc_df = pd.DataFrame(acc_rows)
print("Clock-reading accuracy (% of responses conveying the correct time):")
print(f"{'Model':<25} {'N':>5} {'Image':>8} {'Text':>8} {'Gap':>8}")
print("-" * 60)
for _, row in acc_df.iterrows():
    print(f"{row['model']:<25} {int(row['n']):>5} {row['image']:>7.1%} {row['text']:>7.1%} {row['text']-row['image']:>+7.1%}")

In [ ]:
# ── Figure 1: Modality gap ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

x      = np.arange(len(acc_df))
width  = 0.35
labels = acc_df["model"].tolist()
colors = [MODEL_COLORS[k] for k in acc_df["key"]]

bars_img  = ax.bar(x - width/2, acc_df["image"],  width, label="Image task",  color=colors, alpha=0.55, edgecolor="white")
bars_text = ax.bar(x + width/2, acc_df["text"],   width, label="Text task",   color=colors, alpha=1.00, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Correct time conveyed", fontsize=11)
ax.set_title("Modality gap: image vs. text clock reading (n=475)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

# Annotate bars
for bar in list(bars_img) + list(bars_text):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.0%}",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("modality_gap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Pragmatic shift: full dataset ─────────────────────────────────────────────
# Metric: rate of digital time format (":" present) as a proxy for precision.
# Humans use digital format more often in the police context than the neighbor
# context. We check whether models replicate this shift.

shift_rows = []
for key, label in MODELS.items():
    col = f"{key}_task1_text"
    df  = results.dropna(subset=[col]).copy()
    df["digital"] = df[col].apply(is_digital)
    police   = df[df["context"] == "police"]["digital"].mean()
    neighbor = df[df["context"] == "neighbor"]["digital"].mean()
    shift_rows.append({"model": label, "key": key,
                       "police": police, "neighbor": neighbor,
                       "shift": police - neighbor})

# Human baseline (from full HF dataset)
df_full["digital"] = df_full["production"].apply(is_digital)
h_police   = df_full[df_full["context_str"] == "police"]["digital"].mean()
h_neighbor = df_full[df_full["context_str"] == "neighbor"]["digital"].mean()
shift_rows.append({"model": "Human baseline", "key": "human",
                   "police": h_police, "neighbor": h_neighbor,
                   "shift": h_police - h_neighbor})

shift_df = pd.DataFrame(shift_rows)
print("Digital time rate (precision proxy) — text task, full dataset:")
print(f"{'Model':<25} {'Police':>8} {'Neighbor':>8} {'Shift':>8}")
print("-" * 55)
for _, r in shift_df.iterrows():
    print(f"{r['model']:<25} {r['police']:>7.1%} {r['neighbor']:>7.1%} {r['shift']:>+7.1%}")

In [ ]:
# ── Figure 2: Pragmatic shift ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: police vs neighbor rates per model
ax = axes[0]
x  = np.arange(len(shift_df))
w  = 0.35
bar_colors = [MODEL_COLORS.get(r["key"], "#888888") for _, r in shift_df.iterrows()]

ax.bar(x - w/2, shift_df["police"],   w, label="Police",   color=bar_colors, alpha=1.0,  edgecolor="white")
ax.bar(x + w/2, shift_df["neighbor"], w, label="Neighbor", color=bar_colors, alpha=0.45, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(shift_df["model"], fontsize=9, rotation=15, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Digital time rate", fontsize=11)
ax.set_title("Precision by context (text task)", fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

# Right: pragmatic shift (police − neighbor)
ax = axes[1]
shift_colors = [MODEL_COLORS.get(r["key"], "#888888") for _, r in shift_df.iterrows()]
bars = ax.bar(x, shift_df["shift"], color=shift_colors, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(shift_df["model"], fontsize=9, rotation=15, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel("Pragmatic shift (police − neighbor)", fontsize=11)
ax.set_title("Pragmatic shift: does context affect precision?", fontsize=11, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

for bar, val in zip(bars, shift_df["shift"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.005 if val >= 0 else -0.015),
            f"{val:+.1%}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.savefig("pragmatic_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Wasserstein distance against human baseline ───────────────────────────────
# human_production_code encodes the precision level of each production.
# We compare the police vs. neighbor distributions for humans and for each model
# (using digital/non-digital as a binary proxy for model productions).

human_police_codes   = df_full[df_full["context_str"] == "police"]["production_code"].dropna().astype(float)
human_neighbor_codes = df_full[df_full["context_str"] == "neighbor"]["production_code"].dropna().astype(float)
human_wd = wasserstein_distance(human_police_codes, human_neighbor_codes)

print("Wasserstein distance: police vs. neighbor production distributions")
print(f"(Higher = more context-sensitive; human baseline = {human_wd:.4f})\n")
print(f"{'Source':<25} {'WD':>8}  {'vs. human':>12}")
print("-" * 50)
print(f"{'Human baseline':<25} {human_wd:>8.4f}  {'(reference)':>12}")

for key, label in MODELS.items():
    col = f"{key}_task1_text"
    df  = results.dropna(subset=[col]).copy()
    # Binary production code: 0 = digital (precise), 1 = non-digital (imprecise)
    df["code"] = df[col].apply(lambda r: 0.0 if is_digital(r) else 1.0)
    m_police   = df[df["context"] == "police"]["code"].values
    m_neighbor = df[df["context"] == "neighbor"]["code"].values
    wd = wasserstein_distance(m_police, m_neighbor)
    pct_human = wd / human_wd * 100
    print(f"{label:<25} {wd:>8.4f}  {pct_human:>10.1f}%")

In [ ]:
# ── Per-condition breakdown ───────────────────────────────────────────────────
# Do models fail equally across all clock conditions, or are some harder?
# Compare image accuracy per target_time condition.

cond_rows = []
for key, label in MODELS.items():
    img_col = f"{key}_task1_image"
    df = results.dropna(subset=[img_col]).copy()
    df["correct"] = df.apply(lambda r: time_correct(r[img_col], r["target_time"]), axis=1)
    for tt, grp in df.groupby("target_time"):
        cond_rows.append({"model": label, "target_time": tt, "accuracy": grp["correct"].mean()})

cond_df = pd.pivot_table(
    pd.DataFrame(cond_rows),
    index="target_time", columns="model", values="accuracy"
).round(2)

print("Image task accuracy per clock condition:")
display(cond_df.style.format("{:.0%}").background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1))

### Key takeaways

**Modality gap confirmed across all 475 rows.** All three models read clock *text descriptions* far more accurately than clock *images*. GPT-4o mini and Claude Haiku show a large gap; Gemini 2.5 Flash is substantially better on the image task but still well below text performance — especially for off-round times.

**Pragmatic shift is near zero for all models.** Humans adjust their precision to context: they use digital times more often for the police than for a neighbor (positive shift). All three models show essentially no shift — their precision strategy is fixed regardless of who they are talking to. The Wasserstein distance between police and neighbor distributions is close to zero for all models, versus ~0.27 for humans.

**Condition-level breakdown.** The hardest image conditions are off-round times (8:27, 8:31, 8:33) and range conditions (8:26–8:34). The easiest is 8:30 (half past), which all models recognize. This mirrors what VLM vision research would predict: salience of landmark times biases clock reading toward the nearest round value.

## 8. Open Challenges

**Modality gap — analog clock reading.** All three models read clocks more accurately from text descriptions than from images. GPT-4o mini and Claude Haiku frequently default to "eight o'clock" regardless of what the clock shows. Gemini 2.5 Flash is notably better but still makes errors on off-round times. *Open challenge: better visual grounding for analog clock reading in VLMs.*

**Pragmatic precision calibration.** Models produce digital times at roughly the same rate for police and neighbor contexts. Humans clearly round in casual contexts ("around half past eight") and give exact times in formal ones ("8:28"). Models appear to apply a fixed precision strategy independent of audience. *Open challenge: context-sensitive precision calibration.*

**Task 2 — motive elicitation.** The motive explanations generated by models are fluent and plausible, but they are not yet scored against the human motive annotations. Developing a scoring scheme for pragmatic motive attribution is an open contribution opportunity.

**Open-weight VLMs.** As of mid-2026, benchmarking open-weight VLMs via hosted APIs is non-trivial: most capable models either require dedicated endpoints or have reasoning/thinking modes that interfere with short-response tasks. A local inference setup (e.g. `ollama`, `vllm`) is recommended for open-weight evaluation.

## 9. Extending the Benchmark

**Run the full dataset:**
```bash
python evaluate.py --model gpt-4o-mini --full
```

**Add a new model (OpenAI-compatible):**
```bash
python evaluate.py --model google/gemini-2.5-pro --rows 50  # pilot
python evaluate.py --model google/gemini-2.5-pro --full     # full run
```

**Add a new model (Anthropic):**
```bash
python evaluate.py --model claude-sonnet-4-6 --full
```

Results are merged into `results.csv` automatically — new model columns are added without overwriting existing ones.

**Compute Wasserstein distance** against the human baseline:
```python
from scipy.stats import wasserstein_distance
# human_production_code encodes precision: 0=exact, higher=more imprecise
police   = df[df.context=="police"]["human_production_code"].astype(float)
neighbor = df[df.context=="neighbor"]["human_production_code"].astype(float)
print(wasserstein_distance(police, neighbor))  # human baseline: ~0.27
```